# 00 · Datos de clase — `raw` → `data/processed/`

Este cuaderno es el **primer eslabón** de la secuencia: construye los 8 parquet
y `auditoria_nulos.json` desde el HDF5 DS02. **No contiene la receta**: la
receta vive en `src/clase/dataset_clase.py` y este cuaderno solo la invoca.

**Pipeline (orden exacto):**

1. `build_clase_frame` — une dev + test.
2. `inyectar_nulos_mar` — MAR ~2 % en T30, solo crucero alto (`alt > 27_000 & Mach > 0.68`, seed 7).
3. `split_por_unidad` — train/test por motor (11/14/15 fuera de train).
4. `imputar_ffill` — causal por unidad + mediana **solo de train**.
5. `asignar_fase_vuelo` — Ascenso / Crucero / Descenso desde `alt` y `Mach`.

**Comando canónico:** `python -m src.clase.dataset_clase`

**Verificar sin escribir:** `python -m src.clase.dataset_clase --check`

> `data/processed/` es *build output* y no se versiona. El builder es idempotente:
> si el contenido no cambió, no reescribe.


In [1]:
# Bootstrap: raíz del repo y src/ en sys.path (mismo patrón que NB01-NB07)
import pathlib as _pl, os as _os, sys as _sys
_ROOT = _pl.Path.cwd().resolve()
while not (_ROOT / "pyproject.toml").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_os.chdir(_ROOT)
for _p in (str(_ROOT), str(_ROOT / "src")):
    if _p not in _sys.path:
        _sys.path.insert(0, _p)
_os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
_os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

from src.clase.dataset_clase import construir_artefactos_clase

res = construir_artefactos_clase()

print("filas:", res["filas"])
print("unidades:", res["unidades"])
audit = res["auditoria"]
print("MAR:", audit["mecanismo"], "| nulos:", audit["nulos_inyectados"],
      "| elegibles:", audit["filas_elegibles"],
      "| tasa global T30:", round(audit["tasa_global_T30"], 4))
print("rachas >3:", audit["unidades_con_mas_de_3_seguidos"])
print("escritos:", res["escritos"] or "ninguno (ya estaba al día)")


filas: {'train_con_nulos.parquet': 52635, 'test_con_nulos.parquet': 12538, 'train.parquet': 52635, 'test.parquet': 12538, 'X_train.parquet': 52635, 'X_test.parquet': 12538, 'y_train.parquet': 52635, 'y_test.parquet': 12538}
unidades: {'train': [2.0, 5.0, 10.0, 16.0, 18.0, 20.0], 'test': [11.0, 14.0, 15.0]}
MAR: MAR-altitud | nulos: 1185 | elegibles: 12499 | tasa global T30: 0.0182
rachas >3: 0
escritos: ninguno (ya estaba al día)


**Qué mirar.** `escritos: ninguno` significa que los artefactos ya existían y
el contenido coincide (idempotencia). Los números de referencia son **52.635**
filas de train, **12.538** de test, **1.185** nulos MAR y **12.499** filas
elegibles.

**Siguiente:** `01_eda_y_contexto.ipynb`. Si cambias la inyección de nulos o los
umbrales, borra `data/processed/` y vuelve a ejecutar este cuaderno. Para
comprobar sin escribir nada: `python -m src.clase.dataset_clase --check`.
